# 03. 모델별 전체 Val/Test 및 결과표

선택한 모델 하나만 독립 평가한다. 학습의 official readout을 유지한 체크포인트에 all_targets 추론을 적용한다. 아래 실행은 전체 split이므로 시간이 걸린다. 역사적 결과는 마지막 셀에서 재학습 없이 열람한다.

In [ ]:
from pathlib import Path
import json, subprocess, sys
ROOT = Path.cwd().resolve()
assert (ROOT / "run.py").exists(), "이 공유 폴더에서 노트북을 열어 주세요."
WORK_DIR = ROOT / "work"
RUN_NAME = "shared_seed42"
def run_cli(*args):
    command = [sys.executable, str(ROOT / "run.py"), "--work-dir", str(WORK_DIR), "--run-name", RUN_NAME, *map(str, args)]
    # Pipe and flush each line so progress is visible in Jupyter.
    with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as child:
        try:
            for line in child.stdout:
                print(line, end="", flush=True)
            if child.wait() != 0:
                raise RuntimeError(f"실행 실패: {child.returncode}; 위 로그를 확인하세요.")
        except KeyboardInterrupt:
            child.terminate()
            child.wait()
            raise


In [ ]:
VARIANT = "fraudgt_format"
SCOPE = "test"  # "val"도 별도 실행 가능
CHECKPOINT = None  # 기본: work/runs/<run>/models/<variant>/best.pt
DEVICE = "cuda"
RUN_EVALUATE = False
if RUN_EVALUATE:
    args = ["evaluate", "--variant", VARIANT, "--scope", SCOPE, "--device", DEVICE]
    if CHECKPOINT is not None:
        args += ["--checkpoint", str(CHECKPOINT)]
    run_cli(*args)
else:
    print("RUN_EVALUATE=True이면 선택 모델의 전체 split을 평가합니다.")

In [ ]:
import pandas as pd
print("기존 완료 실험: 전체 Test, all_targets, 비율은 0~1")
display(pd.read_csv(ROOT / "results/full_test_summary.csv"))
print("학습 중 모델 선택에 사용한 조건부 Val")
display(pd.read_csv(ROOT / "results/selected_validation.csv"))
print("전체 Test의 in-degree별 결과")
display(pd.read_csv(ROOT / "results/degree_all_targets.csv"))

임계값/Top-K는 results/threshold_topk.csv, 원본 집계 해시와 논문 비교는 FraudGT_구현_결과_논문비교.md를 참조한다. 현재 PE는 ego ID를 사용하지 않는다.